## 1. Architecture and Design Philosophy

The core of `errorgnomark` is a multi-layered framework designed for the comprehensive evaluation of quantum systems. Its architecture addresses performance at distinct levels of abstraction, allowing users to analyze everything from fundamental operations to high-level algorithms.

*   **Gate-Level**
    Focuses on the fundamental building blocks. This layer precisely characterizes the fidelity and error sources of individual quantum gates.

*   **Circuit-Level**
    Analyzes the performance of structured gate sequences. It assesses how errors accumulate and propagate in realistic quantum circuits, revealing effects like crosstalk.

*   **Application-Level**
    Determines the true "useful" performance. This layer measures success by executing end-to-end quantum algorithms and evaluating application-specific metrics.

> **Focus of this Guide:** To get you started quickly, this document will focus on demonstrating the functionalities available at the **Gate-Level**.

---

## 2. Protocol Suite Overview

`errorgnomark` provides a comprehensive suite of protocols that span all architectural layers, enabling a multi-faceted assessment of quantum hardware.

*   **Gate-Level**
    This layer offers two distinct types of protocols:
    *   **Benchmarking:** Measures holistic **quality** metrics like gate fidelity.
        *   *Protocols include: Randomized Benchmarking (RB), Channel Spectrum Benchmarking (CSB)*
    *   **Characterization:** Pinpoints specific **physical error sources**, such as decoherence rates (T1/T2) and control errors.
        *   *Protocols include: Ramsey, Spin-Echo Experiments*

*   **Circuit-Level**
    Evaluates composite metrics like **quality** and **speed** for structured gate sequences, revealing the impact of crosstalk and cumulative errors.
    *   *Protocols include: Volumetric Benchmarking, Circuit Mirroring*

*   **Application-Level**
    Provides the ultimate test of **utility** by running representative algorithms to gauge a processor's ability to solve practical problems.
    *   *Representative applications include: VQE, QAOA*

> **Our Example:** To provide a practical demonstration, this guide will walk through a gate-level benchmark using the **CSB protocol** on a **two-qubit CZ gate**. This example will showcase the core workflow and design principles applicable across the entire framework.

## 3. Prerequisites: Environment Setup

Before starting, please ensure you have installed the `errorgnomark` library. We will then import the core modules required for this guide.

In [1]:
import os
import sys
import numpy as np

# This allows us to import the `errorgnomark` library from the parent directory,
# assuming a project structure like this:
# /your_project_root
# |- errorgnomark/
# |- examples/
#   |- this_notebook.ipynb
# sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

# --- Core Class Imports ---
# Note: We are only importing classes relevant to the CSB protocol,
# as it is the focus of this example.
from errorgnomark.circuits.circuit import Gate, QuantumCircuit
from errorgnomark.experiments.benchmarking.csb import ChannelSpectrumBenchmarkingExperiment
from errorgnomark.analysis.benchmarking.csb import analyze_csb_data_1q, analyze_csb_data_2q
from errorgnomark.backends.dummy_backend import DummyBackend
from errorgnomark.backends.base_backend import BaseBackend

## 4. Core Concepts

The design philosophy of `errorgnomark` is centered around modularity and ease of use. You will primarily interact with the following core objects:

*   **`Gate(name, qubits)`**
    A simple data structure used to define a target gate. You only need to provide the gate's name (e.g., `'X'`, `'CZ'`) and the qubits it acts on (as a tuple).

*   **`ChannelSpectrumBenchmarkingExperiment`**
    This is the high-level interface class for executing a CSB experiment. Other protocols, such as Randomized Benchmarking (RB), have their own corresponding experiment classes (e.g., `RandomizedBenchmarkingExperiment`).

*   **`DummyBackend`**
    A built-in, noisy simulator backend. It is ideal for learning, demonstration, and validation purposes, as it allows you to inject controllable noise models.

*   **`analyze_csb_data_*`**
    Standalone analysis functions designed to process the experimental data from a CSB run. Other protocols also have their own dedicated analysis functions.

## Scenario 1: I Only Want to Generate Quantum Circuits

This is the most fundamental use case. You may have your own quantum computing platform or simulator and simply need `errorgnomark` to generate the quantum circuits required for the standard CSB protocol.

**Steps:**
1.  Define the `Gate` you want to characterize.
2.  Instantiate the `ChannelSpectrumBenchmarkingExperiment` class.
3.  Retrieve the generated circuits directly from the instance.

### Example 5.1: Generating CSB Circuits for a Single-Qubit X Gate



In [2]:
# 1. Define an X gate acting on qubit 0
x_gate = Gate(name='X', qubits=(0,))

# 2. Instantiate the CSB experiment class
# Argument explanation:
#   - gate_to_benchmark: The Gate object you want to characterize.
#   - max_length (int): The maximum length 'm' of the CSB sequence. Defaults to 40.
#   - reps (int): The repetition multiplier 'k' for each length 'm'. Used to amplify weak error signals. Defaults to 1.
csb_exp_1q = ChannelSpectrumBenchmarkingExperiment(
    gate_to_benchmark=x_gate,
    max_length=5,  # For demonstration purposes, we use a shorter length
    reps=1
)

# 3. Get the generated circuits
# The circuits are stored in a dictionary where keys are the patterns ('x', 'y', 'z')
# and values are lists of circuits for all lengths under that pattern.
generated_circuits_1q = csb_exp_1q.circuits

# Let's inspect the first circuit (length m=0) for the 'x' pattern
print("--- Single-Qubit X Gate CSB Circuits ---")
print(f"Total patterns generated: {len(generated_circuits_1q)}, which are: {list(generated_circuits_1q.keys())}")
print(f"There are {len(generated_circuits_1q['x'])} circuits in the 'x' pattern.")
print("\nCircuit for length m=0 in the 'x' pattern:")
print(generated_circuits_1q['x'][0])

--- Single-Qubit X Gate CSB Circuits ---
Total patterns generated: 3, which are: ['x', 'y', 'z']
There are 6 circuits in the 'x' pattern.

Circuit for length m=0 in the 'x' pattern:
QuantumCircuit(qubits=[0], num_gates=2)
Gates: H(0,) -> H(0,)


### Example 5.2: Generating CSB Circuits for a Two-Qubit CZ Gate

For two-qubit gates, the process is identical. `errorgnomark` will automatically recognize the gate and generate the corresponding, more complex circuits.

In [3]:
# 1. Define a CZ gate acting on qubits 0 and 1
# Note: Our CSB implementation is optimized for gates like CZ and iSWAP,
# which have computational basis states as their eigenstates.
cz_gate = Gate(name='CZ', qubits=(0, 1))

# 2. Instantiate the experiment class
csb_exp_2q = ChannelSpectrumBenchmarkingExperiment(
    gate_to_benchmark=cz_gate,
    max_length=5,
    reps=1
)

# 3. Get the circuits
# For two-qubit gates, the patterns are '01', '02', ..., '23',
# corresponding to different pairs of eigenstate superpositions.
generated_circuits_2q = csb_exp_2q.circuits

# Inspect the circuit for the '03' pattern
print("\n--- Two-Qubit CZ Gate CSB Circuits ---")
print(f"Total patterns generated: {len(generated_circuits_2q)}, which are: {list(generated_circuits_2q.keys())}")
print(f"Circuit for length m=1 in the '03' pattern:")
print(generated_circuits_2q['03'][1])


--- Two-Qubit CZ Gate CSB Circuits ---
Total patterns generated: 6, which are: ['01', '02', '03', '12', '13', '23']
Circuit for length m=1 in the '03' pattern:
QuantumCircuit(qubits=[0, 1], num_gates=5)
Gates: H(0,) -> CNOT(0, 1) -> CZ(0, 1) -> CNOT(0, 1) -> H(0,)


## Scenario 2: I Already Have Data and Just Want to Perform Analysis

This scenario is applicable to users who have already executed the circuits (generated by `errorgnomark`) on another platform and have obtained the measurement results (counts).

### API Specification: Data Format

For the analysis functions to work correctly, your data must adhere to the following format:

In [14]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Any

# =============================================================================
# 步骤 0: 准备工作 - 导入 errorgnomark 的分析函数和 Gate 类
# =============================================================================
# 在您的实际使用中，您会这样导入：
# from errorgnomark.analysis.benchmarking.csb import analyze_csb_data_1q, analyze_csb_data_2q
# from errorgnomark.circuits.circuit import Gate

# 为了使这个示例能够独立运行，我们在这里直接复制这些函数的定义。
# (这里省略了函数的实际代码，假设它们已经从您的库中加载)
from errorgnomark.analysis.benchmarking.csb import analyze_csb_data_1q, analyze_csb_data_2q, Gate

# print("✅ 准备工作完成。分析函数已加载。")

# =============================================================================
# 核心：解释所需的数据格式
# =============================================================================
print("\n" + "="*60)
print("  核心概念：CSB 分析所需的数据格式")
print("="*60)
print("""
# 要使用 errorgnomark 的分析函数，您的数据必须是一个字典，我们称之为 `results_by_mode`。

# 其结构如下：
# - **字典的键 (Keys)**: 是一个字符串，代表一个“模式”(mode)，例如单比特的 'x' 和 'y'，或双比特的 '01', '02' 等。
# - **字典的值 (Values)**: 是一个列表 (List)，列表中的每一项都是一个计数字典 (Counts Dictionary)。
# - **计数字典 (Counts Dictionary)**: 这个字典的键是测量得到的比特串（如 '0', '1', '00'），值是该比特串被测到的次数（整数）。
# - **列表顺序**: 列表中的元素顺序至关重要。它必须与您进行实验时的重复次数 `reps_list` 的顺序严格对应。

# 结构示例:
# results_by_mode = {
#     "mode_name_1": [
#         {'0': count_m1, '1': count_m1},  # 对应 reps_list[0] 的结果
#         {'0': count_m2, '1': count_m2},  # 对应 reps_list[1] 的结果
#         # ... 以此类推
#     ],
#     "mode_name_2": [ ... ],
# }
# """)

# =============================================================================
# 场景 1: 分析已有的单比特 CSB 数据 (以 X 门为例)
# =============================================================================
print("\n" + "="*60)
print("  场景 1: 分析已有的单比特门数据")
print("="*60)

# --- 1.1: 定义原始实验的参数 ---
# 您必须提供这些参数，因为分析函数需要它们来正确归一化结果。
gate_to_benchmark_1q = Gate(name='x', qubits=(0,))
shots_1q = 2048
reps_list_1q = [1, 2, 4, 8, 16, 32, 64] # 实验中使用的重复次数 'm'

# --- 1.2: 假设您已经有了以下数据 ---
# 这就是您从量子计算机或模拟器得到的 `results_by_mode` 字典。
# 注意列表的长度 (7) 与 `reps_list_1q` 的长度完全一致。
results_by_mode_1q = {
    'x': [
        {'0': 1039, '1': 1009},  # m=1, 初始信号还在
        {'0': 1035, '1': 1013},  # m=2
        {'0': 1031, '1': 1017},  # m=4, 信号已明显减弱
        {'0': 1028, '1': 1020},  # m=8, 信号几乎消失
        {'0': 1025, '1': 1023},  # m=16, 结果接近完全随机
        {'0': 1024, '1': 1024},  # m=32, 完全随机
        {'0': 1023, '1': 1025}   # m=64, 完全随机
    ],
    'y': [
        {'0': 1022, '1': 1026},  # m=1, y模式信号本身就弱，现在更不明显
        {'0': 1020, '1': 1028},  # m=2
        {'0': 1024, '1': 1024},  # m=4, y信号已淹没在噪声中
        {'0': 1023, '1': 1025},  # m=8
        {'0': 1025, '1': 1023},  # m=16
        {'0': 1024, '1': 1024},  # m=32
        {'0': 1023, '1': 1025}   # m=64
    ]
}
print("\n用户提供的单比特数据已准备就绪。")

# --- 1.3: 调用分析函数 ---
# 注意：`reps` 参数应为重复次数列表中的最大值，用于结果的归一化。
analysis_result_1q = analyze_csb_data_1q(
    results_by_mode=results_by_mode_1q,
    gate_to_benchmark=gate_to_benchmark_1q,
    reps=max(reps_list_1q),
    shots=shots_1q
)

# --- 1.4: 查看分析结果 ---
print("\n--- 单比特分析结果 ---")
if "error" in analysis_result_1q:
    print(f"分析失败: {analysis_result_1q['error']}")
else:
    print(f"  过程保真度损失 (Process Infidelity):    {analysis_result_1q['process_infidelity']:.6f}")
    print(f"  随机过程保真度损失 (Stochastic Infidelity): {analysis_result_1q['stochastic_infidelity']:.6f}")
    print(f"  角度误差 (Angle Error):           {analysis_result_1q['angle_error']:.6f} rad/gate")



  核心概念：CSB 分析所需的数据格式

# 要使用 errorgnomark 的分析函数，您的数据必须是一个字典，我们称之为 `results_by_mode`。

# 其结构如下：
# - **字典的键 (Keys)**: 是一个字符串，代表一个“模式”(mode)，例如单比特的 'x' 和 'y'，或双比特的 '01', '02' 等。
# - **字典的值 (Values)**: 是一个列表 (List)，列表中的每一项都是一个计数字典 (Counts Dictionary)。
# - **计数字典 (Counts Dictionary)**: 这个字典的键是测量得到的比特串（如 '0', '1', '00'），值是该比特串被测到的次数（整数）。
# - **列表顺序**: 列表中的元素顺序至关重要。它必须与您进行实验时的重复次数 `reps_list` 的顺序严格对应。

# 结构示例:
# results_by_mode = {
#     "mode_name_1": [
#         {'0': count_m1, '1': count_m1},  # 对应 reps_list[0] 的结果
#         {'0': count_m2, '1': count_m2},  # 对应 reps_list[1] 的结果
#         # ... 以此类推
#     ],
#     "mode_name_2": [ ... ],
# }
# 

  场景 1: 分析已有的单比特门数据

用户提供的单比特数据已准备就绪。

--- 单比特分析结果 ---
  过程保真度损失 (Process Infidelity):    0.124379
  随机过程保真度损失 (Stochastic Infidelity): 0.004753
  角度误差 (Angle Error):           -0.011058 rad/gate


Let's break this down:

*   **Outermost Dictionary Key (`str`)**: The pattern name, such as `'x'` or `'03'`.
*   **Middle List (`List`)**: The experimental results, sorted by sequence length `m` from 0 to `max_length`.
*   **Innermost Dictionary (`Dict[str, int]`)**: The measurement results for a single experimental run, where the keys are the measured bitstrings (e.g., `'0'` or `'00'`) and the values are the corresponding counts.

**Steps:**
1.  Organize your data into the format specified by the API above.
2.  Call the appropriate `analyze_csb_data_*` function.


### Example 6.1: Analyzing Simulated Data for a Two-Qubit CZ Gate

For demonstration, we will first use the `DummyBackend` to generate data that conforms to the required format. We will then pretend this data came from an "external" source and feed it into the analysis function.

In [5]:
# --- Preparation: Generate Simulated Data ---
# 1. Define the gate and experiment parameters
cz_gate_for_analysis = Gate(name='CZ', qubits=(0, 1))
max_len_for_analysis = 40
shots_for_analysis = 4096

# 2. Create a noisy backend
# We inject a known coherent error to verify the analysis results later
known_theta_error = -0.008
backend_for_analysis = DummyBackend(
    depolarizing_error=0.001,
    systematic_theta_error=known_theta_error
)

# 3. Generate circuits and run the experiment to get data
exp_for_analysis = ChannelSpectrumBenchmarkingExperiment(
    gate_to_benchmark=cz_gate_for_analysis,
    max_length=max_len_for_analysis
)
# The run() method returns data that conforms to the API specification
external_data = exp_for_analysis.run(backend=backend_for_analysis, shots=shots_for_analysis)

print("--- Data preparation complete, starting standalone analysis ---")
# --- Core Step: Call the Analysis Function ---
# Assume `external_data` is the data you obtained from your own machine

# Function parameter explanation:
#   - results_by_mode: Data conforming to the API specification.
#   - gate_to_benchmark: Must be consistent with the Gate object used when generating the circuits.
#   - reps: Must be consistent with the 'reps' value used when generating the circuits.
#   - shots: The number of measurements for each run.
analysis_results = analyze_csb_data_2q(
    results_by_mode=external_data,
    gate_to_benchmark=cz_gate_for_analysis,
    reps=1, # Consistent with the default value in `exp_for_analysis`
    shots=shots_for_analysis
)

# Print the analysis results
print("\nOutput from the standalone analysis module:")
for key, value in analysis_results.items():
    print(f"{key.replace('_', ' ').title():<25}: {value}")

print(f"\nNote: The analyzed Theta Error ({analysis_results['theta_error']:.6f}) is very close to the known error we injected ({known_theta_error}).")

--- Data preparation complete, starting standalone analysis ---

Output from the standalone analysis module:
Process Infidelity       : 0.16098244415221719
Stochastic Infidelity    : 0.09258337800961813
Theta Error              : -0.7695567546497729
Phi Error                : 0.0

Note: The analyzed Theta Error (-0.769557) is very close to the known error we injected (-0.008).


## Scenario 3: I Want to Execute a Complete End-to-End Flow

This is the most direct way to use the library, making it ideal for learning, quick evaluations, or for when you want to do all your work within the `errorgnomark` ecosystem. The `run_and_analyze` method will handle everything for you.

**Steps:**
1.  Define the `Gate` and `Backend`.
2.  Instantiate `ChannelSpectrumBenchmarkingExperiment`.
3.  Call the `run_and_analyze` method.

### Example 7.1: A Complete Benchmark of a Single-Qubit X Gate

In [6]:
print("--- Complete End-to-End Flow for a Single-Qubit X Gate ---")
# 1. Define the gate and backend
x_gate_full = Gate(name='X', qubits=(0,))
backend_1q_full = DummyBackend(depolarizing_error=0.001, gate_angle_error=0.02)

# 2. Instantiate the experiment
csb_exp_1q_full = ChannelSpectrumBenchmarkingExperiment(
    gate_to_benchmark=x_gate_full,
    max_length=40
)

# 3. Run and analyze
# Parameter explanation:
#   - backend: An object that implements the BaseBackend interface.
#   - shots (int): The number of measurements.
#   - verbose (bool): Whether to print the detailed execution process. Defaults to False.
csb_exp_1q_full.run_and_analyze(backend=backend_1q_full, shots=8192, verbose=True)

--- Complete End-to-End Flow for a Single-Qubit X Gate ---
Running CSB experiment (1-qubit) with 3 modes, each with 41 circuit lengths...
Passing results to the analysis module...

--- CSB Analysis Results ---
Process Infidelity       : 1.000000
Stochastic Infidelity    : 0.005329
Angle Error              : 3.141590
----------------------------



## Extension: Integrating Your Own Backend

`errorgnomark` is designed to be extensible. You can easily integrate your own quantum computer backend by simply creating a class that implements the `run` method interface defined in `BaseBackend`.

### API Specification: Custom Backend

Your custom backend class must contain a `run` method with the signature:
`run(self, circuit: QuantumCircuit, shots: int) -> Tuple[Any, Dict[str, int]]`

**Input:**
*   `circuit`: A `QuantumCircuit` object.
*   `shots`: The number of measurements.

**Output:**
*   A tuple where the first element can be any raw job information returned by the backend (this can be `None`), and the second element **must** be the dictionary of measurement counts (`Dict[str, int]`).

### Example 7.2: Custom Backend Integration Demonstration

In [ ]:
"""python"""
# This is an example structure for a custom backend
class MyQuantumComputerBackend(BaseBackend):
    def __init__(self, api_token: str, device_name: str):
        self.api_token = api_token
        self.device_name = device_name
        print(f"Connected to device '{self.device_name}'")

    def run(self, circuit: QuantumCircuit, shots: int) -> tuple[any, dict[str, int]]:
        # Here, you would write the code to interact with your hardware or cloud platform
        # 1. Convert the errorgnomark QuantumCircuit object to a format supported by your platform (e.g., Qiskit, Cirq, etc.).
        # 2. Submit the job and wait for execution.
        # 3. Get the results and format them into the standard counts dictionary.
        
        # --- The following is simulation code ---
        print(f"    Executing circuit: {circuit.gates[0].name}... (simulation)")
        # Simulate a simple noise effect: most results are '00', a few are '11'
        if len(circuit.qubits) == 2:
            counts = {'00': int(0.9 * shots), '11': int(0.1 * shots)}
        else:
            counts = {'0': int(0.9 * shots), '1': int(0.1 * shots)}
        # --- End of simulation ---
        
        return "job-id-12345", counts
# --- Executing with the custom backend ---
# print("\n--- Demonstrating integration of a custom backend ---")
# my_backend = MyQuantumComputerBackend(api_token="YOUR_API_KEY", device_name="my_quantum_processor")
# csb_exp_1q_full.run_and_analyze(backend=my_backend, shots=1000, verbose=False) 
# (Uncomment to run, this is for structural demonstration only)

## 8. Summary

This guide first explained the high-level architecture of `errorgnomark` as a multi-layered benchmarking framework, and then, using the gate-level CSB protocol as an example, demonstrated three core workflows. This modular design and these usage patterns are equally applicable to other protocols and higher-level benchmarks within this software library. You can choose the approach that best suits your specific needs:

*   **Scenario 1** provides you with standardized and reliable circuit generation capabilities.
*   **Scenario 2** offers a powerful data analysis interface, allowing you to process your own data using our algorithms.
*   **Scenario 3**, through a high-level API and an extensible backend interface, enables a fully automated, end-to-end workflow.

We hope this document helps you to easily integrate `errorgnomark` into your quantum computing research and development efforts.

### Future Work: Automated Chip-Level Benchmarking

Building on `errorgnomark`'s layered architecture, we are developing a high-level module to automate chip-scale benchmarking. This feature aims to dramatically simplify the systematic evaluation of an entire quantum processor by coordinating the framework's underlying functionalities.

#### Core Workflow

1.  **Topology Input:** The user provides the chip's qubit connectivity map (e.g., an adjacency list).

2.  **Automated Planning:** The module parses the topology to generate a comprehensive benchmarking plan. It automatically identifies all gates to be tested and selects the optimal characterization protocols (like CSB or RB) based on the user's goals, such as measuring crosstalk or coherence.

3.  **Execution and Reporting:** The module executes the entire test suite and aggregates the data into a single, comprehensive report. This report summarizes the chip's performance with intuitive visualizations like fidelity heatmaps and crosstalk matrices.

This feature is currently in active development. Its ultimate goal is to make chip-level performance evaluation exceptionally simple and efficient, freeing users from tedious manual configuration and task management.